# Feature Engineering Phase 2: Past-Only Behavioral Features

This experiment tests historical counts, amounts, recency, and velocity with fixed LightGBM parameters. Every feature uses only rows earlier in deterministic `(TransactionDT, TransactionID)` order. Velocity additionally requires a strictly earlier timestamp. Validation is processed sequentially from TRAIN state; labels never enter feature state. The reserved final split is not assigned, transformed, inspected, or evaluated.

In [1]:
import gc
import sys
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
    validate_training_data,
)
from fraud_detection.features import BehavioralFeatureEngineer, Phase1FeatureEngineer
from fraud_detection.models import TrainingOnlyPreprocessor, evaluate_probabilities

plt.style.use("seaborn-v0_8-whitegrid")
notebook_started = time.perf_counter()

## Methodology and leakage boundary

Cumulative features sort by `TransactionDT`, then `TransactionID`; the current row is shifted out. For velocity at time `T`, only timestamps satisfying `T - window < previous_time < T` count, so equal-time events are excluded. Missing entities use an explicit history bucket. TRAIN initializes state, and each validation row can use earlier validation rows only after their features have been calculated.

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validation = split.validation
validate_training_data(train)
del transaction, identity, merged
gc.collect()

excluded_columns = ["isFraud", "TransactionID"]
feature_columns = [column for column in train.columns if column not in excluded_columns]
train_predictors = train.loc[:, feature_columns]
validation_predictors = validation.loc[:, feature_columns]
y_train = train["isFraud"].astype("int8")
y_validation = validation["isFraud"].astype("int8")

print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Raw baseline feature count: {len(feature_columns)}")
print("Reserved final split status: not assigned or accessed")

Train rows: 413,378
Validation rows: 88,581
Raw baseline feature count: 432
Reserved final split status: not assigned or accessed


In [3]:
feature_started = time.perf_counter()
frequency_engineer = Phase1FeatureEngineer(
    include_amount_time=False,
    include_missing_indicators=False,
    include_frequency_encodings=True,
).fit(train_predictors)
frequency_train = frequency_engineer.transform(train_predictors)
frequency_validation = frequency_engineer.transform(validation_predictors)

behavior_source_columns = [
    column
    for column in (
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "card1",
        "addr1",
        "DeviceInfo",
        "P_emaildomain",
    )
    if column in train.columns
]
behavioral_engineer = BehavioralFeatureEngineer()
behavioral_train = behavioral_engineer.fit_transform_features(train.loc[:, behavior_source_columns])
behavioral_validation = behavioral_engineer.transform_features(
    validation.loc[:, behavior_source_columns], update_state=True
)
feature_generation_seconds = time.perf_counter() - feature_started
behavioral_memory_mb = (
    behavioral_train.memory_usage(deep=True).sum()
    + behavioral_validation.memory_usage(deep=True).sum()
) / 1024**2

print(f"Frequency-only feature count: {frequency_train.shape[1]}")
print(f"Behavioral features generated: {behavioral_train.shape[1]}")
print(f"All Phase 2 feature count: {frequency_train.shape[1] + behavioral_train.shape[1]}")
print(f"Behavioral generation seconds: {feature_generation_seconds:.1f}")
print(f"Behavioral matrices memory: {behavioral_memory_mb:.1f} MiB")
display(pd.Series(behavioral_engineer.feature_groups_, name="features"))

Frequency-only feature count: 438
Behavioral features generated: 14
All Phase 2 feature count: 452
Behavioral generation seconds: 7.8
Behavioral matrices memory: 26.8 MiB


historical_counts     (card1_previous_tx_count, addr1_previous_tx_co...
historical_amounts    (card1_previous_avg_amount, card1_previous_max...
recency               (seconds_since_card1_previous_transaction, sec...
velocity              (card1_tx_count_last_1h, card1_tx_count_last_24h)
Name: features, dtype: object

In [4]:
feature_sets = {
    "raw_baseline": None,
    "frequency_only": (),
    "plus_historical_counts": behavioral_engineer.feature_groups_["historical_counts"],
    "plus_historical_amounts": behavioral_engineer.feature_groups_["historical_amounts"],
    "plus_recency": behavioral_engineer.feature_groups_["recency"],
    "plus_velocity": behavioral_engineer.feature_groups_["velocity"],
    "all_phase2": behavioral_engineer.generated_feature_names_,
}
lightgbm_parameters = dict(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-behavioral-features")
feature_sets

2026/10/06 10:47:20 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-behavioral-features' does not exist. Creating a new experiment.


{'raw_baseline': None,
 'frequency_only': (),
 'plus_historical_counts': ('card1_previous_tx_count',
  'addr1_previous_tx_count',
  'device_previous_tx_count',
  'email_previous_tx_count',
  'card1_addr1_previous_tx_count'),
 'plus_historical_amounts': ('card1_previous_avg_amount',
  'card1_previous_max_amount',
  'amount_vs_card1_previous_avg',
  'addr1_previous_avg_amount',
  'amount_vs_addr1_previous_avg'),
 'plus_recency': ('seconds_since_card1_previous_transaction',
  'seconds_since_device_previous_transaction'),
 'plus_velocity': ('card1_tx_count_last_1h', 'card1_tx_count_last_24h'),
 'all_phase2': ('card1_previous_tx_count',
  'addr1_previous_tx_count',
  'device_previous_tx_count',
  'email_previous_tx_count',
  'card1_addr1_previous_tx_count',
  'card1_previous_avg_amount',
  'card1_previous_max_amount',
  'amount_vs_card1_previous_avg',
  'addr1_previous_avg_amount',
  'amount_vs_addr1_previous_avg',
  'seconds_since_card1_previous_transaction',
  'seconds_since_device_previo

In [5]:
rows = []
for feature_set_name, behavioral_columns in feature_sets.items():
    if behavioral_columns is None:
        experiment_train = train_predictors
        experiment_validation = validation_predictors
    elif not behavioral_columns:
        experiment_train = frequency_train
        experiment_validation = frequency_validation
    else:
        experiment_train = pd.concat(
            [frequency_train, behavioral_train.loc[:, list(behavioral_columns)]], axis=1
        )
        experiment_validation = pd.concat(
            [
                frequency_validation,
                behavioral_validation.loc[:, list(behavioral_columns)],
            ],
            axis=1,
        )

    model_started = time.perf_counter()
    preprocessor = TrainingOnlyPreprocessor()
    x_train = preprocessor.fit_transform(experiment_train)
    x_validation = preprocessor.transform(experiment_validation)
    scaler = StandardScaler(copy=False)
    x_train = scaler.fit_transform(x_train)
    x_validation = scaler.transform(x_validation)

    with mlflow.start_run(run_name=f"behavioral-{feature_set_name}") as run:
        model = lgb.LGBMClassifier(**lightgbm_parameters)
        model.fit(x_train, y_train)
        probabilities = model.predict_proba(x_validation)[:, 1]
        metrics = evaluate_probabilities(y_validation, probabilities)
        model_seconds = time.perf_counter() - model_started
        mlflow.set_tags(
            {
                "stage": "behavioral-feature-engineering",
                "dataset": "ieee-cis",
                "split": "temporal-validation",
            }
        )
        mlflow.log_params(
            {
                "feature_set_name": feature_set_name,
                "feature_count": x_train.shape[1],
                "learning_rate": lightgbm_parameters["learning_rate"],
                "n_estimators": lightgbm_parameters["n_estimators"],
                "num_leaves": lightgbm_parameters["num_leaves"],
            }
        )
        mlflow.log_metrics(
            {
                "pr_auc": metrics.pr_auc,
                "roc_auc": metrics.roc_auc,
                "precision_at_0_50": metrics.precision,
                "recall_at_0_50": metrics.recall,
                "f1_at_0_50": metrics.f1,
                "model_seconds": model_seconds,
            }
        )
        rows.append(
            {
                "Version": feature_set_name,
                "Feature Count": x_train.shape[1],
                "PR-AUC": metrics.pr_auc,
                "ROC-AUC": metrics.roc_auc,
                "Precision": metrics.precision,
                "Recall": metrics.recall,
                "F1": metrics.f1,
                "Model Seconds": model_seconds,
                "MLflow Run ID": run.info.run_id,
            }
        )

    del x_train, x_validation, model, probabilities, preprocessor, scaler
    if behavioral_columns:
        del experiment_train, experiment_validation
    gc.collect()

results = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
display(results)

,Version,Feature Count,PR-AUC,ROC-AUC,Precision,Recall,F1,Model Seconds,MLflow Run ID
0,plus_historical_amounts,443,0.593938,0.919463,0.573609,0.555884,0.564608,40.302877,48a204e028e446ca9f7a6b1b645d5cb5
1,plus_recency,440,0.593936,0.924308,0.534618,0.583826,0.558140,64.408778,35f3fc7874154fa39d06caa57b4e985d
2,frequency_only,438,0.589631,0.921839,0.577205,0.544379,0.560311,36.474870,602b1198212d4b30949ef00d4eeb9f4b
3,plus_velocity,440,0.589453,0.922244,0.556248,0.557528,0.556887,39.025897,39bfe63e9bbb418e8258dbe6f3fdcf42
4,raw_baseline,432,0.587317,0.921482,0.572638,0.533859,0.552569,34.210178,84151b90cc7a4b44b33fd67bf2a7991d
5,all_phase2,452,0.581880,0.921848,0.526961,0.565417,0.545512,41.971928,642c228fa92b470cbcfb63322f6a22fa
6,plus_historical_counts,443,0.581491,0.918272,0.546960,0.564760,0.555717,32.792538,7f4a9a33770c45c4b07ccb31adc1d72b


In [6]:
frequency_result = results.loc[results["Version"] == "frequency_only"].iloc[0]
behavioral_results = results.loc[~results["Version"].isin(["raw_baseline", "frequency_only"])]
best_behavioral = behavioral_results.iloc[0]
absolute_improvement = best_behavioral["PR-AUC"] - frequency_result["PR-AUC"]
relative_improvement = absolute_improvement / frequency_result["PR-AUC"]
total_seconds = time.perf_counter() - notebook_started

figure_path = PROJECT_ROOT / "reports" / "figures" / "behavioral_ablation.png"
plot_data = results.sort_values("PR-AUC")
fig, axis = plt.subplots(figsize=(9, 5.5))
axis.barh(plot_data["Version"], plot_data["PR-AUC"], color="#4C78A8")
axis.axvline(
    frequency_result["PR-AUC"],
    color="#E45756",
    linestyle="--",
    label="Frequency-only",
)
axis.set(
    title="Past-only behavioral feature ablation",
    xlabel="Validation PR-AUC",
    ylabel="Feature set",
)
axis.legend()
fig.tight_layout()
fig.savefig(figure_path, dpi=160)
plt.close(fig)


def markdown_results_table(frame):
    header = "| Version | Features | PR-AUC | ROC-AUC | Precision | Recall | F1 | Seconds |"
    divider = "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |"
    table_rows = []
    for _, row in frame.iterrows():
        table_rows.append(
            f"| {row['Version']} | {int(row['Feature Count'])} | "
            f"{row['PR-AUC']:.6f} | {row['ROC-AUC']:.6f} | "
            f"{row['Precision']:.6f} | {row['Recall']:.6f} | "
            f"{row['F1']:.6f} | {row['Model Seconds']:.1f} |"
        )
    return "\n".join([header, divider, *table_rows])


feature_lines = "\n".join(
    f"- **{group}:** {', '.join(features)}"
    for group, features in behavioral_engineer.feature_groups_.items()
)
result_word = "improvement" if absolute_improvement > 0 else "degradation"
helpful_groups = behavioral_results.loc[
    behavioral_results["PR-AUC"] > frequency_result["PR-AUC"], "Version"
].tolist()
unhelpful_groups = behavioral_results.loc[
    behavioral_results["PR-AUC"] <= frequency_result["PR-AUC"], "Version"
].tolist()
report = f"""# Past-Only Behavioral Features

## Implemented features

{feature_lines}

## Past-only methodology and leakage safeguards

Rows are ordered by `TransactionDT`, then `TransactionID`. Counts, amount aggregates, and
recency are calculated before the current row updates history. Velocity requires
`T - window < previous_time < T`, excluding current and equal-time transactions. Missing
entities use an explicit bucket. TRAIN alone initializes state; VALIDATION is processed
sequentially without labels. No target statistic is used. The reserved final split is untouched.

## Observed validation results

{markdown_results_table(results)}

- Previous best frequency-only PR-AUC: **{frequency_result["PR-AUC"]:.6f}**
- Best behavioral feature set: **{best_behavioral["Version"]}**
- New best behavioral PR-AUC: **{best_behavioral["PR-AUC"]:.6f}**
- Absolute PR-AUC {result_word}: **{absolute_improvement:+.6f}**
- Relative PR-AUC {result_word}: **{relative_improvement:+.2%}**
- Winning behavioral model features: **{int(best_behavioral["Feature Count"])}**

![Behavioral ablation](figures/behavioral_ablation.png)

## Runtime and memory observations

- Feature generation: **{feature_generation_seconds:.1f} seconds**
- Behavioral feature matrices: **{behavioral_memory_mb:.1f} MiB**
- Notebook experiment time before report write: **{total_seconds:.1f} seconds**
- Window counts use grouped NumPy binary searches rather than full-data row loops.

## Interpretation of observed results

- Helped versus frequency-only: **{", ".join(helpful_groups)}**.
- Hurt or added little value: **{", ".join(unhelpful_groups)}**.
Feature groups are retained only when their validation PR-AUC exceeds the frequency-only
reference. Threshold metrics remain fixed at 0.50 and were not optimized.

## Hypotheses for later work (not implemented)

Entity-specific window choices or carefully selected additional entities may improve signal,
but require separate leakage-safe validation. Graph features, rules, SHAP, and threshold
optimization remain outside this phase.
"""
report_path = PROJECT_ROOT / "reports" / "behavioral_features_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Absolute PR-AUC change: {absolute_improvement:+.6f}")
print(f"Relative PR-AUC change: {relative_improvement:+.2%}")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Figure: {figure_path.relative_to(PROJECT_ROOT)}")

Absolute PR-AUC change: +0.004307
Relative PR-AUC change: +0.73%
Report: reports\behavioral_features_summary.md
Figure: reports\figures\behavioral_ablation.png


## Scope boundary

This notebook stops after Phase 2 validation impact measurement. It does not implement graph features, rules, SHAP, threshold optimization, retraining, serving, or deployment.